# Weighted OD Matrices by Mode — CAR / TRANSIT / OTHER

Recreates the household-weighted Day 10 / Day 20 matrices from `THS_2018_MTX_weighted.ipynb`, split by an aggregated travel mode. `MODE_NAME` is mapped to three groups:

| Group | Modes |
|---|---|
| **CAR** | Vehicle as Driver, Vehicle as Passenger, Motorcycle/Moped |
| **TRANSIT** | Public Bus, Matronit, Train, Special Taxi, Group Taxi |
| **OTHER** | Pedestrian, Default, Chartered Bus, Bicycle or other personal means, Other, Truck |

Trip extraction is unchanged (order by `INDIVID`/`tourID`/`ACT_ID`, origin = previous activity's `taz`, leaving time = `EndTime` for Home / `StartTime` otherwise, departures 6:00–9:00, each trip contributes its household's `wf_new`). **A trip's mode is the `MODE_NAME` of its destination activity row** — the mode used to arrive at the current activity.

In [1]:
import numpy as np
import pandas as pd

MODE_GROUP = {
    'Vehicle as Driver': 'CAR',
    'Pedestrian': 'OTHER',
    'Default': 'OTHER',
    'Public Bus': 'TRANSIT',
    'Vehicle as Passenger': 'CAR',
    'Matronit': 'TRANSIT',
    'Train': 'TRANSIT',
    'Special Taxi': 'TRANSIT',
    'Group Taxi': 'TRANSIT',
    'Chartered Bus': 'OTHER',
    'Motorcycle/Moped': 'CAR',
    'Bicycle or other personal means': 'OTHER',
    'Other': 'OTHER',
    'Truck': 'OTHER',
}
MODES = ['CAR', 'TRANSIT', 'OTHER']

In [2]:
df = pd.read_csv('Input/Matrices/ACTIVITIES_DEC18_corrected.csv')
weights = pd.read_csv('Input/Matrices/households_with_weights.csv')[['HHID', 'wf_new']]
df = df.merge(weights, on='HHID', how='left', validate='many_to_one')
assert df['wf_new'].notna().all()

unmapped = set(df['MODE_NAME'].dropna().unique()) - set(MODE_GROUP)
assert not unmapped, f"MODE_NAME values missing from the dictionary: {unmapped}"
assert df['MODE_NAME'].notna().all()
df['mode_group'] = df['MODE_NAME'].map(MODE_GROUP)
print("mode-group distribution over all activity records:")
print(df['mode_group'].value_counts().to_string())

mode-group distribution over all activity records:
mode_group
CAR        88328
OTHER      73158
TRANSIT    11043


In [3]:
def am_peak_trips(df_day):
    d = df_day.sort_values(by=['INDIVID', 'tourID', 'ACT_ID'])
    d['origin'] = d.groupby(['INDIVID', 'tourID'])['taz'].shift(1)
    d['destination'] = d['taz']
    d['StartTime'] = pd.to_datetime(d['StartTime'], dayfirst=True)
    d['EndTime'] = pd.to_datetime(d['EndTime'], dayfirst=True)
    d['leaving_time'] = pd.to_datetime(np.where(d['mainActivity'] == 'Home', d['EndTime'], d['StartTime']))
    mask = (d['leaving_time'].dt.hour >= 6) & (d['leaving_time'].dt.hour < 9)
    # a trip's mode = MODE_NAME of the destination (current) activity row
    return d[mask].dropna(subset=['origin', 'destination'])

matrices = {}
summary_rows = []
for day, df_day in [(10, df[df['ACT_DAY'] == 10]), (20, df[df['ACT_DAY'] == 20])]:
    trips = am_peak_trips(df_day.copy())
    for mode in MODES:
        t = trips[trips['mode_group'] == mode]
        m = pd.crosstab(t['origin'], t['destination'], values=t['wf_new'], aggfunc='sum').fillna(0)
        matrices[(day, mode)] = m
        summary_rows.append({'day': day, 'mode': mode,
                             'sampled trips': len(t),
                             'expanded trips': t['wf_new'].sum()})

summary = pd.DataFrame(summary_rows).set_index(['day', 'mode'])
summary['share'] = summary['expanded trips'] / summary.groupby('day')['expanded trips'].transform('sum')
summary.round({'expanded trips': 0, 'share': 3})

sampled trips  expanded trips  share
day mode                                         
10  CAR               7942       1298688.0  0.565
    TRANSIT           1129        164217.0  0.071
    OTHER             5841        833913.0  0.363
20  CAR               7792       1263355.0  0.558
    TRANSIT           1139        167447.0  0.074
    OTHER             5774        833356.0  0.368

In [4]:
# consistency check: per day, the three mode matrices must sum to the all-mode weighted total
for day, ref_total in [(10, 2296819.0), (20, 2264157.5)]:
    total = sum(matrices[(day, mode)].sum().sum() for mode in MODES)
    assert abs(total - ref_total) < 1.0, (day, total)
    print(f"day {day}: CAR + TRANSIT + OTHER = {total:,.1f} expanded trips — matches the all-mode matrix")

day 10: CAR + TRANSIT + OTHER = 2,296,819.0 expanded trips — matches the all-mode matrix
day 20: CAR + TRANSIT + OTHER = 2,264,157.5 expanded trips — matches the all-mode matrix


In [5]:
import os
os.makedirs('Output', exist_ok=True)
for (day, mode), m in matrices.items():
    path = f'Output/matrix_{day}_weighted_{mode}.csv'
    m.to_csv(path)
    print(f"{path}:  shape {m.shape},  {m.sum().sum():,.0f} expanded trips")

Output/matrix_10_weighted_CAR.csv:  shape (621, 676),  1,298,688 expanded trips
Output/matrix_10_weighted_TRANSIT.csv:  shape (337, 336),  164,217 expanded trips


Output/matrix_10_weighted_OTHER.csv:  shape (563, 608),  833,913 expanded trips


Output/matrix_20_weighted_CAR.csv:  shape (610, 665),  1,263,355 expanded trips
Output/matrix_20_weighted_TRANSIT.csv:  shape (344, 336),  167,447 expanded trips


Output/matrix_20_weighted_OTHER.csv:  shape (573, 603),  833,356 expanded trips


## Notes

- The mode split applies to the *arriving* leg of each activity transition. `Default` (typically activities with no reported travel, e.g. the diary's opening record) falls under OTHER per the dictionary; such rows rarely survive the trip filter since a trip requires a previous activity in the same tour.
- Matrices only include origin/destination zones observed for that day-mode combination, so shapes differ across modes; align with `.reindex(...)` before comparing.
- The three matrices per day sum exactly to the corresponding all-mode weighted matrix (`matrix_10_weighted.csv` / `matrix_20_weighted.csv`).